<a name='1'></a>
## Résumer un dialogue sans Prompt Engineering

Dans ce cas d’utilisation, vous allez générer un résumé d’un dialogue avec le modèle de langage pré-entraîné (LLM) Qwen3.5-0.8B de Hugging Face. La liste des modèles disponibles dans le package transformers de Hugging Face se trouve [ici](https://huggingface.co/docs/transformers/index).

Chargeons maintenant quelques dialogues simples issus du dataset [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) de Hugging Face. Ce dataset contient plus de 10 000 dialogues accompagnés de leurs résumés manuellement annotés ainsi que de leurs thématiques.


In [ ]:
from datasets import load_dataset
from transformers import pipeline


In [ ]:
huggingface_dataset_name = "knkarthick/dialogsum"
dataset = load_dataset(huggingface_dataset_name)

In [ ]:
example_indices = [40, 200]

dash_line = '-'.join('' for x in range(100))

for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print('INPUT DIALOGUE:')
    print(dataset['test'][index]['dialogue'])
    print(dash_line)
    print('BASELINE HUMAN SUMMARY:')
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print()

In [ ]:
# Chargement du modele preentraine Qwen3.5-0.8B (comme dans le notebook RAG).
# On le charge UNE seule fois, puis on reutilise le pipeline `pipe`.
pipe = pipeline("image-text-to-text", model="Qwen/Qwen3.5-0.8B")


def generate(prompt, max_new_tokens=50):
    """Genere du texte avec Qwen3.5-0.8B a partir d'un prompt (str)."""
    messages = [
        {"role": "user", "content": [{"type": "text", "text": prompt}]},
    ]
    history = pipe(text=messages, max_new_tokens=max_new_tokens)
    return history[-1]['generated_text'][-1]['content']


In [ ]:
# Sans prompt engineering
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']

    output = generate(dialogue, max_new_tokens=50)

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{dialogue}')
    print(dash_line)

    summary = dataset['test'][index]['summary']
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')

    print(dash_line)
    print(f'MODEL GENERATION - WITHOUT PROMPT ENGINEERING:\n{output}\n')


In [ ]:
# Minimal prompt engineering -> c'est a peine mieux
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue'] + '\nSummary:'
    summary = dataset['test'][index]['summary']

    output = generate(dialogue, max_new_tokens=50)

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{dialogue}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')
    print(dash_line)
    print(f'MODEL GENERATION - WITHOUT PROMPT ENGINEERING:\n{output}\n')


In [ ]:
# Exercice 1: Faire du prompt engineering afin d'ameliorer la sortie. Pour de meilleures performances, prompter en anglais !
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
        """

    output = generate(prompt, max_new_tokens=50)

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{prompt}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')
    print(dash_line)
    print(f'MODEL GENERATION - ZERO SHOT:\n{output}\n')


In [ ]:
# Exercice 2: Faire du prompt engineering + One-Shot
def make_prompt(example_indices_full, example_index_to_summarize):
    prompt = ''
    for index in example_indices_full:
        dialogue = dataset['test'][index]['dialogue']
        summary = dataset['test'][index]['summary']

        prompt += f"""
Dialogue:

{dialogue}

What was going on?
{summary}


"""

    dialogue = dataset['test'][example_index_to_summarize]['dialogue']

    prompt += f"""
Dialogue:

{dialogue}

What was going on?
"""

    return prompt


example_indices_full = [40]
example_index_to_summarize = 200

one_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

print(one_shot_prompt)


In [ ]:
summary = dataset['test'][example_index_to_summarize]['summary']

output = generate(one_shot_prompt, max_new_tokens=50)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ONE SHOT:\n{output}')


In [ ]:
# Exercice 3: Faire du prompt engineering + Few-Shot. A partir de combien d'exemples il est inutile d'en ajouter ?

example_indices_full = [40, 80, 120]
example_index_to_summarize = 200

few_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

print(few_shot_prompt)

In [ ]:
summary = dataset['test'][example_index_to_summarize]['summary']

output = generate(few_shot_prompt, max_new_tokens=50)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')


# Exercice : Classification de sentiments par prompt engineering

Jusqu'ici on a *résumé*. On va maintenant *classifier* : prédire le sentiment
(`positive` / `negative`) d'un avis, **sans entraînement**, uniquement par le prompt.
On compare deux régimes d'*in-context learning* :

- **Zero-shot** : on décrit la tâche, aucun exemple.
- **Few-shot** : on fournit quelques exemples résolus dans le prompt.

On réutilise la fonction `generate` (Qwen3.5-0.8B) déjà définie plus haut. Comme indiqué,
le modèle répond mieux **en anglais**.


In [ ]:
# Petit jeu de test etiquete (equilibre). Verite terrain = `label`.
reviews = [
    ("I absolutely loved this movie, it was brilliant.", "positive"),
    ("A complete waste of time, terrible acting.",        "negative"),
    ("Best purchase I have made this year, highly recommend.", "positive"),
    ("The product broke after two days, very disappointing.",  "negative"),
    ("Fantastic experience, I would come back again.",    "positive"),
    ("Awful service and the food was cold.",              "negative"),
    ("This book changed my life, truly inspiring.",       "positive"),
    ("Boring, predictable and far too long.",             "negative"),
]


def accuracy(predict_fn):
    ok = 0
    for text, label in reviews:
        pred = predict_fn(text).strip().lower()
        norm = "positive" if "pos" in pred else ("negative" if "neg" in pred else pred)
        ok += (norm == label)
        print(f"{label:8} | pred={pred:10} | {text[:45]}")
    print(f"\nAccuracy : {ok}/{len(reviews)} = {100*ok/len(reviews):.0f}%")
    return ok / len(reviews)


## Zero-shot

**À compléter :** rédigez un prompt *zero-shot* qui demande au modèle de classer l'avis
en `positive` ou `negative`.

In [ ]:
def predict_zero_shot(text):
    prompt = (
        "Classify the sentiment of the following review as positive or negative.\n"
        f"Review: {text}\n"
        "Sentiment:"
    )
    return generate(prompt)

print("=== ZERO-SHOT ===")
acc_zero = accuracy(predict_zero_shot)


## Few-shot

**À compléter :** ajoutez 2 à 4 exemples résolus *avant* l'avis à classer (few-shot).
Observez l'effet sur l'accuracy et sur la *stabilité* du format de sortie.

In [ ]:
def predict_few_shot(text):
    prompt = (
        "Classify the sentiment of each review as positive or negative.\n"
        "Review: The plot was amazing and the cast was perfect.\nSentiment: positive\n"
        "Review: I regret buying this, it stopped working immediately.\nSentiment: negative\n"
        "Review: What a delightful and heartwarming story.\nSentiment: positive\n"
        f"Review: {text}\nSentiment:"
    )
    return generate(prompt)

print("=== FEW-SHOT ===")
acc_few = accuracy(predict_few_shot)
print(f"\nGain few-shot vs zero-shot : {100*(acc_few-acc_zero):+.0f} points")
